# ARC-v0.32 — NQ-GTE Exhaustive-Reference Audit

**Scientific role:** post-primary, reviewer-oriented audit for the SIGIR full-paper claim that a *relative* higher-fidelity ANN comparator is sufficient to expose mechanism-conditioned retrieval-in-the-loop behavior.

This notebook does **not** retune ARC-v0.27/0.28. It reuses their Natural Questions + GTE setup and asks a narrower question:

> Do the short-horizon representation-vs-search-effort trajectory conclusions remain qualitatively similar when the common reference is an exhaustive Flat inner-product search over the same persisted corpus-embedding store?

### Important reference definition

The persisted NQ corpus store consists of **2,681,468 normalized 384-d vectors saved as float16 blocks**. The Flat reference below casts those stored vectors to float32 and performs exhaustive inner-product search with no ANN pruning or vector quantizer.

Therefore this is:

- **exact/exhaustive with respect to the persisted shared corpus-embedding store**, but
- **not** an oracle for the encoder's original pre-persistence FP32 outputs.

That distinction is deliberately retained in every artifact and manuscript-facing string.

### Frozen design

- Dataset: BEIR Natural Questions
- Encoder: `thenlper/gte-small`
- Representation low branch: IVF-PQ32, `nprobe=64`
- Relative high branch: IVF-SQ8, `nprobe=64`
- Search-effort low branch: IVF-SQ8, `nprobe=2`
- Exhaustive reference: FlatIP over the persisted shared corpus store
- Horizon: `H=4`
- State operator: anchored centroid feedback
- Main subset: the **already frozen 500 ARC-v0.31 main query IDs**
- Policy subset: the **outcome-independent eight-policy structural subset** already used by ARC-v0.28 long-horizon work (`mean-k20` and `softmax-k20-t0.1` for each alpha in `{0.1,0.3,0.5,0.7}`)
- Primary inference: 10,000 paired-query bootstrap replicates after averaging policies within query
- All positive, null, reversed, or qualitatively different outcomes are retained.


In [ ]:
# Cell 1 — Colab / FAISS setup
import os, sys, subprocess

def run(cmd):
    return subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, check=False)

gpu_present = run(["bash", "-lc", "nvidia-smi -L"]).returncode == 0

# Match the v0.28 environment as closely as practical.
run([sys.executable, "-m", "pip", "uninstall", "-y", "faiss-cpu", "faiss-gpu", "faiss-gpu-cu12"])
common = ["pyarrow", "scipy", "tqdm", "psutil", "requests"]

if gpu_present:
    p = run([sys.executable, "-m", "pip", "install", "-q", "faiss-gpu-cu12==1.14.1.post1", *common])
    print(p.stdout[-2000:])
    if p.returncode != 0:
        print("GPU FAISS wheel unavailable; falling back to faiss-cpu.")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu", *common])
else:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu", *common])

from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import gc, hashlib, json, math, random, shutil, time, zipfile

import faiss
import numpy as np
import pandas as pd
import requests
from scipy.stats import spearmanr
from tqdm.auto import tqdm
from google.colab import drive

SEED = 20260832
V027_SPLIT_SEED = 20260827
DIM = 384
TOP_RETRIEVE = 100
UTILITY_K = 10
H = 4
BOOTSTRAP_REPS = 10_000
REP_NPROBE = 64
SEARCH_LOW_NPROBE = 2
SEARCH_HIGH_NPROBE = 64
EXPECTED_N = 3452
EXPECTED_VAL = 1726
EXPECTED_VAL_SHA = "8548189d7a7e5fda6150642f5e6eed4215baa04d4baf389c99acf632ae6e04cf"

random.seed(SEED)
np.random.seed(SEED)
if not gpu_present:
    faiss.omp_set_num_threads(os.cpu_count() or 1)

print("faiss:", getattr(faiss, "__version__", "unknown"))
print("GPU visible:", gpu_present)


In [ ]:
# Cell 2 — Drive paths and immutable source artifacts
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
ARC_ROOT = DRIVE_ROOT / 'rag-pq-checkpoints' / 'arc-v0'
LARGE_ROOT = DRIVE_ROOT / 'rag-pq-checkpoints' / 'arc-v027-nq-gte-large-cache'
V031_ROOT = ARC_ROOT / 'generative-retrieval-agent-transfer-v031'
V032_ROOT = ARC_ROOT / 'nq-gte-exhaustive-reference-v032'
V032_ROOT.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
OUT = V032_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

PQ_PATH = LARGE_ROOT / 'nq-gte-ivfpq-nlist4096-m32-nbits8.faiss'
SQ_PATH = LARGE_ROOT / 'nq-gte-ivfsq8-nlist4096.faiss'
QUERY_IDS_PATH = LARGE_ROOT / 'nq_query_ids.txt'
QUERY_EMB_PATH = LARGE_ROOT / 'nq_query_embeddings.float32.npy'
CORPUS_MANIFEST_PATH = LARGE_ROOT / 'nq_corpus_block_manifest.json'
QREL_ROW_MAP_PATH = LARGE_ROOT / 'nq_qrel_doc_rows.csv'
V031_PROTOCOL_PATH = V031_ROOT / 'V031_FROZEN_PROTOCOL.json'

required = [PQ_PATH, SQ_PATH, QUERY_IDS_PATH, QUERY_EMB_PATH, CORPUS_MANIFEST_PATH, QREL_ROW_MAP_PATH, V031_PROTOCOL_PATH]
missing = [str(p) for p in required if not p.is_file()]
assert not missing, f"Missing source artifacts: {missing}"
print('OUT:', OUT)


In [ ]:
# Cell 3 — Provenance helpers and source validation
def sha256_file(path, chunk=16 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

def membership_sha(ids):
    return hashlib.sha256('\n'.join(sorted(map(str, ids))).encode('utf-8')).hexdigest()

def normalize_rows(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float32)
    n = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(n, eps)

manifest = json.loads(CORPUS_MANIFEST_PATH.read_text())
assert manifest['status'] == 'COMPLETE'
assert manifest['encoder'] == 'thenlper/gte-small'
assert int(manifest['dimension']) == DIM
assert int(manifest['corpus_rows']) == 2_681_468
assert len(manifest['blocks']) == 27

ALL_QUERY_IDS = QUERY_IDS_PATH.read_text(encoding='utf-8').splitlines()
assert len(ALL_QUERY_IDS) == EXPECTED_N
query_embeddings = np.load(QUERY_EMB_PATH, mmap_mode='r')
assert query_embeddings.shape == (EXPECTED_N, DIM)
qid_to_pos = {q:i for i,q in enumerate(ALL_QUERY_IDS)}

def split_key(qid):
    return hashlib.sha256(f'{V027_SPLIT_SEED}|{qid}'.encode('utf-8')).hexdigest()
ordered = sorted(ALL_QUERY_IDS, key=split_key)
cut = len(ordered)//2
FIT_IDS, VAL_IDS = ordered[:cut], ordered[cut:]
assert len(VAL_IDS) == EXPECTED_VAL
assert membership_sha(VAL_IDS) == EXPECTED_VAL_SHA

v031 = json.loads(V031_PROTOCOL_PATH.read_text())
MAIN_IDS = list(map(str, v031['main_ids']))
assert len(MAIN_IDS) == 500 and len(set(MAIN_IDS)) == 500
assert set(MAIN_IDS).issubset(set(VAL_IDS))

POLICIES = []
for alpha in [0.1, 0.3, 0.5, 0.7]:
    POLICIES.append({'name': f'mean-k20-a{alpha}', 'family':'mean', 'k':20, 'tau':None, 'alpha':alpha})
    POLICIES.append({'name': f'softmax-k20-t0.1-a{alpha}', 'family':'softmax', 'k':20, 'tau':0.1, 'alpha':alpha})
assert len(POLICIES) == 8

print('validation queries:', len(VAL_IDS))
print('frozen main queries:', len(MAIN_IDS))
print('policies:', [p['name'] for p in POLICIES])


In [ ]:
# Cell 4 — Freeze v0.32 protocol BEFORE new outcomes
protocol = {
    'study_id': 'ARC-v0.32',
    'status': 'FROZEN_BEFORE_EXHAUSTIVE_REFERENCE_OUTCOMES',
    'scientific_role': 'post-primary reviewer-oriented exact/exhaustive-reference audit',
    'dataset': 'beir/nq',
    'encoder': 'thenlper/gte-small',
    'reference_definition': (
        'Exhaustive FlatIP over the persisted shared normalized corpus-embedding store. '
        'Source corpus blocks are float16 and are cast to float32 for exhaustive dot products; '
        'this is exact w.r.t. the persisted store, not an oracle for pre-persistence encoder FP32 outputs.'
    ),
    'H': H,
    'operator': 'anchored',
    'search_k': TOP_RETRIEVE,
    'utility_k': UTILITY_K,
    'representation': {'low':'IVF-PQ32', 'low_nprobe':REP_NPROBE, 'relative_high':'IVF-SQ8', 'high_nprobe':SEARCH_HIGH_NPROBE},
    'search_effort': {'low':'IVF-SQ8', 'low_nprobe':SEARCH_LOW_NPROBE, 'relative_high':'IVF-SQ8', 'high_nprobe':SEARCH_HIGH_NPROBE},
    'main_subset_source': 'ARC-v0.31 frozen main_ids',
    'n_main_queries': len(MAIN_IDS),
    'main_ids_sha256': membership_sha(MAIN_IDS),
    'validation_membership_sha256': membership_sha(VAL_IDS),
    'policy_subset_source': 'ARC-v0.28 outcome-independent structural H50 subset',
    'policies': POLICIES,
    'bootstrap_reps': BOOTSTRAP_REPS,
    'primary': 'representation_minus_search_effort_H3abs_using_exhaustive_reference',
    'primary_direction': 'two-sided',
    'secondary': [
        'relative-high vs exhaustive-reference change in cross-mechanism H3abs contrast',
        'residual IVF-SQ8-nprobe64 vs exhaustive-reference one-shot and trajectory gaps',
        'terminal absolute-gap levels',
        'full-1726-query round-0 exact-reference candidate/effectiveness audit',
    ],
    'retention_rule': 'Retain positive, null, reversed, or qualitatively different outcomes; no comparator/policy/query/horizon/endpoint retuning after this freeze.',
    'source_hashes': {
        'corpus_manifest_sha256': sha256_file(CORPUS_MANIFEST_PATH),
        'pq_index_sha256': sha256_file(PQ_PATH),
        'sq_index_sha256': sha256_file(SQ_PATH),
        'query_embeddings_sha256': sha256_file(QUERY_EMB_PATH),
        'v031_protocol_sha256': sha256_file(V031_PROTOCOL_PATH),
    },
}
protocol_path = OUT / 'V032_FROZEN_PROTOCOL.json'
protocol_path.write_text(json.dumps(protocol, indent=2, sort_keys=True), encoding='utf-8')
protocol_sha = sha256_file(protocol_path)
(OUT/'V032_PROTOCOL_SHA256.txt').write_text(protocol_sha+'  V032_FROZEN_PROTOCOL.json\n')
print('V032 protocol SHA-256:', protocol_sha)


In [ ]:
# Cell 5 — Localize corpus blocks (recommended for feedback-vector gathers)
LOCALIZE_BLOCKS = True
LOCAL_BLOCK_ROOT = Path('/content/arc-v032-nq-corpus-blocks')
LOCAL_BLOCK_ROOT.mkdir(parents=True, exist_ok=True)

block_records = manifest['blocks']
BLOCK_PATHS = []
for rec in tqdm(block_records, desc='Corpus blocks'):
    src = Path(rec['path'])
    assert src.is_file(), src
    if LOCALIZE_BLOCKS:
        dst = LOCAL_BLOCK_ROOT / src.name
        if not dst.is_file() or dst.stat().st_size != src.stat().st_size:
            shutil.copy2(src, dst)
        use = dst
    else:
        use = src
    BLOCK_PATHS.append(use)

# Structural + norm audit on a deterministic sample.
rng = np.random.default_rng(SEED)
sample_parts = []
for p in BLOCK_PATHS[:3]:
    a = np.load(p, mmap_mode='r')
    take = rng.choice(len(a), size=min(1024, len(a)), replace=False)
    sample_parts.append(np.asarray(a[take], dtype=np.float32))
sample = np.vstack(sample_parts)
norms = np.linalg.norm(sample, axis=1)
print(pd.Series(norms).describe())
assert np.isfinite(sample).all()
# Do not silently renormalize the corpus; audit the persisted representation as used by the project.
assert float(np.mean(np.abs(norms - 1.0))) < 5e-3, 'Persisted corpus vectors are unexpectedly far from unit norm.'


In [ ]:
# Cell 6 — Shared vector store for feedback updates
class BlockEmbeddingStore:
    def __init__(self, paths, dim=384, block_rows=100_000):
        self.paths = list(paths)
        self.dim = dim
        self.block_rows = block_rows
        self.cache = {}

    def _block(self, b):
        b = int(b)
        if b not in self.cache:
            self.cache[b] = np.load(self.paths[b], mmap_mode='r')
        return self.cache[b]

    def get(self, ids):
        ids = np.asarray(ids, dtype=np.int64)
        shape = ids.shape
        flat = ids.reshape(-1)
        if np.any(flat < 0) or np.any(flat >= manifest['corpus_rows']):
            raise ValueError('Invalid corpus row id encountered.')
        out = np.empty((len(flat), self.dim), dtype=np.float32)
        blocks = flat // self.block_rows
        for b in np.unique(blocks):
            mask = blocks == b
            local = flat[mask] - int(b) * self.block_rows
            out[mask] = np.asarray(self._block(int(b))[local], dtype=np.float32)
        return out.reshape(*shape, self.dim)

store = BlockEmbeddingStore(BLOCK_PATHS, DIM)
# Smoke parity: rows gathered from block 0 should match direct load.
smoke_ids = np.array([0, 1, 99999, 100000, 2681467])
smoke = store.get(smoke_ids)
assert smoke.shape == (5, DIM) and np.isfinite(smoke).all()
print('Embedding store ready.')


In [ ]:
# Cell 7 — Obtain BEIR NQ qrels and map relevant doc IDs to corpus rows
LOCAL_RAW_ROOT = Path('/content/arc-v032-nq-raw')
LOCAL_RAW_ROOT.mkdir(parents=True, exist_ok=True)
ZIP_PATH = LOCAL_RAW_ROOT/'nq.zip'
NQ_DIR = LOCAL_RAW_ROOT/'nq'
NQ_URL = 'https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/nq.zip'

if not (NQ_DIR/'qrels'/'test.tsv').is_file():
    if not ZIP_PATH.is_file():
        with requests.get(NQ_URL, stream=True, timeout=120) as r:
            r.raise_for_status()
            total = int(r.headers.get('content-length', 0))
            with open(ZIP_PATH, 'wb') as f, tqdm(total=total or None, unit='B', unit_scale=True, desc='nq.zip') as bar:
                for chunk in r.iter_content(8*1024*1024):
                    if chunk:
                        f.write(chunk); bar.update(len(chunk))
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(LOCAL_RAW_ROOT)

qrels_path = NQ_DIR/'qrels'/'test.tsv'
qrels_df = pd.read_csv(qrels_path, sep='\t')
qid_col = next(c for c in ['query-id','query_id','qid'] if c in qrels_df.columns)
doc_col = next(c for c in ['corpus-id','corpus_id','doc_id'] if c in qrels_df.columns)
score_col = next((c for c in ['score','relevance','rel'] if c in qrels_df.columns), None)
qrels_df[qid_col] = qrels_df[qid_col].astype(str)
qrels_df[doc_col] = qrels_df[doc_col].astype(str)
if score_col is not None:
    qrels_df[score_col] = pd.to_numeric(qrels_df[score_col], errors='coerce')
    qrels_df = qrels_df[qrels_df[score_col] > 0].copy()

rel_row_map = pd.read_csv(QREL_ROW_MAP_PATH)
rel_row_map['doc_id'] = rel_row_map['doc_id'].astype(str)
DOC_TO_ROW = dict(zip(rel_row_map['doc_id'], rel_row_map['corpus_row'].astype(int)))
assert set(qrels_df[doc_col]).issubset(DOC_TO_ROW), 'Relevant doc row map is incomplete.'

QRELS = defaultdict(set)
for _, r in qrels_df.iterrows():
    QRELS[str(r[qid_col])].add(int(DOC_TO_ROW[str(r[doc_col])]))
assert set(VAL_IDS).issubset(QRELS), 'Some validation qids have no mapped positive qrels.'
print('qrels queries:', len(QRELS), 'mapped positive docs:', len(DOC_TO_ROW))


In [ ]:
# Cell 8 — Build exhaustive FlatIP reference and load ANN indexes
# The Flat index receives the persisted vectors in corpus-row order. No ANN pruning is used.
# FP32 accumulation is requested on GPU; if GPU Flat construction fails, CPU IndexFlatIP is retained.

cpu_flat = faiss.IndexFlatIP(DIM)
for p in tqdm(BLOCK_PATHS, desc='Building CPU FlatIP'):
    xb = np.asarray(np.load(p, mmap_mode='r'), dtype=np.float32)
    cpu_flat.add(xb)
assert cpu_flat.ntotal == manifest['corpus_rows']

flat_index = cpu_flat
flat_backend = 'CPU_FP32_FLATIP'
gpu_res = None
if gpu_present and hasattr(faiss, 'StandardGpuResources'):
    try:
        gpu_res = faiss.StandardGpuResources()
        co = faiss.GpuClonerOptions()
        co.useFloat16 = False
        flat_index = faiss.index_cpu_to_gpu(gpu_res, 0, cpu_flat, co)
        flat_backend = 'GPU_FP32_FLATIP'
        print('Exhaustive reference moved to GPU in FP32 mode.')
    except Exception as e:
        print('Flat GPU clone failed; keeping CPU FlatIP:', repr(e))

pq_cpu = faiss.read_index(str(PQ_PATH))
sq_cpu = faiss.read_index(str(SQ_PATH))
assert pq_cpu.ntotal == sq_cpu.ntotal == manifest['corpus_rows']

# ANN indexes may stay on CPU if their exact serialized type cannot be faithfully moved to GPU.
pq_index, sq_index = pq_cpu, sq_cpu
ann_backend = 'CPU'
if gpu_present and gpu_res is not None:
    try:
        pq_index = faiss.index_cpu_to_gpu(gpu_res, 0, pq_cpu)
        sq_index = faiss.index_cpu_to_gpu(gpu_res, 0, sq_cpu)
        ann_backend = 'GPU'
    except Exception as e:
        pq_index, sq_index = pq_cpu, sq_cpu
        print('ANN GPU clone not jointly available; using CPU ANN indexes:', repr(e))

print('flat backend:', flat_backend)
print('ANN backend:', ann_backend)


In [ ]:
# Cell 9 — Retrieval, utility, feedback, and slope helpers
def set_nprobe(index, nprobe):
    try:
        index.nprobe = int(nprobe)
    except Exception:
        ps = faiss.ParameterSpace()
        ps.set_index_parameter(index, 'nprobe', int(nprobe))

def batched_search(index, q, k=TOP_RETRIEVE, batch=256, nprobe=None):
    if nprobe is not None:
        set_nprobe(index, nprobe)
    q = np.ascontiguousarray(q, dtype=np.float32)
    Ds, Is = [], []
    for s in range(0, len(q), batch):
        D, I = index.search(q[s:s+batch], k)
        Ds.append(np.asarray(D)); Is.append(np.asarray(I))
    return np.vstack(Ds), np.vstack(Is)

def ndcg10(qids, retrieved):
    out = np.zeros(len(qids), dtype=np.float64)
    discounts = 1.0 / np.log2(np.arange(2, UTILITY_K + 2))
    for i, (qid, ids) in enumerate(zip(qids, retrieved)):
        rel = QRELS[str(qid)]
        gains = np.fromiter((1.0 if int(d) in rel else 0.0 for d in ids[:UTILITY_K]), dtype=np.float64, count=UTILITY_K)
        dcg = float((gains * discounts).sum())
        ideal_n = min(len(rel), UTILITY_K)
        idcg = float(discounts[:ideal_n].sum()) if ideal_n else 0.0
        out[i] = dcg / idcg if idcg > 0 else 0.0
    return out

def feedback(ids, scores, policy):
    k = int(policy['k'])
    V = store.get(ids[:, :k])
    if policy['family'] == 'mean':
        F = V.mean(axis=1)
    elif policy['family'] == 'softmax':
        z = scores[:, :k].astype(np.float64) / float(policy['tau'])
        z -= z.max(axis=1, keepdims=True)
        w = np.exp(z); w /= w.sum(axis=1, keepdims=True)
        F = np.einsum('nk,nkd->nd', w.astype(np.float32), V)
    else:
        raise ValueError(policy)
    return normalize_rows(F)

def anchored_update(q0, F, alpha):
    return normalize_rows((1.0-float(alpha))*q0 + float(alpha)*F)

def ols_slope(y):
    # y: (..., H+1), x = 0..H
    y = np.asarray(y, dtype=np.float64)
    x = np.arange(y.shape[-1], dtype=np.float64)
    xc = x - x.mean()
    return np.tensordot(y, xc, axes=([-1],[0])) / float(np.dot(xc, xc))

def candidate_overlap(a, b, k=100):
    vals = []
    for x,y in zip(a[:,:k], b[:,:k]):
        vals.append(len(set(map(int,x)) & set(map(int,y))) / k)
    return np.asarray(vals)


In [ ]:
# Cell 10 — Full 1,726-query round-0 exhaustive-reference audit
VAL_POS = np.array([qid_to_pos[q] for q in VAL_IDS], dtype=np.int64)
QVAL = normalize_rows(np.asarray(query_embeddings[VAL_POS], dtype=np.float32))

D_exact0, I_exact0 = batched_search(flat_index, QVAL)
D_high0, I_high0 = batched_search(sq_index, QVAL, nprobe=SEARCH_HIGH_NPROBE)
D_search0, I_search0 = batched_search(sq_index, QVAL, nprobe=SEARCH_LOW_NPROBE)
D_rep0, I_rep0 = batched_search(pq_index, QVAL, nprobe=REP_NPROBE)

u_exact0 = ndcg10(VAL_IDS, I_exact0)
u_high0 = ndcg10(VAL_IDS, I_high0)
u_search0 = ndcg10(VAL_IDS, I_search0)
u_rep0 = ndcg10(VAL_IDS, I_rep0)

round0 = pd.DataFrame({
    'query_id': VAL_IDS,
    'ndcg_exact': u_exact0,
    'ndcg_sq8_np64': u_high0,
    'ndcg_sq8_np2': u_search0,
    'ndcg_pq32_np64': u_rep0,
    'sq8_np64_exact_top100_overlap': candidate_overlap(I_high0, I_exact0, 100),
    'sq8_np2_exact_top100_overlap': candidate_overlap(I_search0, I_exact0, 100),
    'pq32_np64_exact_top100_overlap': candidate_overlap(I_rep0, I_exact0, 100),
    'sq8_np64_abs_ndcg_gap_to_exact': np.abs(u_exact0-u_high0),
    'sq8_np2_abs_ndcg_gap_to_exact': np.abs(u_exact0-u_search0),
    'pq32_np64_abs_ndcg_gap_to_exact': np.abs(u_exact0-u_rep0),
})
round0.to_csv(OUT/'v032_round0_full_validation_exact_reference.csv', index=False)
summary0 = round0.drop(columns='query_id').agg(['mean','median','std']).T
print(summary0)
summary0.to_csv(OUT/'v032_round0_full_validation_summary.csv')


In [ ]:
# Cell 11 — Frozen 500-query × 8-policy × H=4 anchored trajectory sweep
MAIN_POS = np.array([qid_to_pos[q] for q in MAIN_IDS], dtype=np.int64)
Q0 = normalize_rows(np.asarray(query_embeddings[MAIN_POS], dtype=np.float32))

# Keep only compact utility trajectories; all branch searches are recomputed from frozen inputs.
records = []
start_time = time.time()

for p_idx, policy in enumerate(POLICIES):
    print(f"[{p_idx+1}/{len(POLICIES)}] {policy['name']}")
    states = {
        'rep_low': Q0.copy(),
        'high': Q0.copy(),
        'search_low': Q0.copy(),
        'exact': Q0.copy(),
    }
    utilities = {k: [] for k in states}

    for t in range(H+1):
        branch_out = {}
        D,I = batched_search(pq_index, states['rep_low'], nprobe=REP_NPROBE)
        branch_out['rep_low'] = (D,I)
        D,I = batched_search(sq_index, states['high'], nprobe=SEARCH_HIGH_NPROBE)
        branch_out['high'] = (D,I)
        D,I = batched_search(sq_index, states['search_low'], nprobe=SEARCH_LOW_NPROBE)
        branch_out['search_low'] = (D,I)
        D,I = batched_search(flat_index, states['exact'])
        branch_out['exact'] = (D,I)

        for b,(D,I) in branch_out.items():
            utilities[b].append(ndcg10(MAIN_IDS, I))

        if t < H:
            for b,(D,I) in branch_out.items():
                F = feedback(I, D, policy)
                states[b] = anchored_update(Q0, F, policy['alpha'])

    # shape n_query x (H+1)
    U = {b: np.stack(v, axis=1) for b,v in utilities.items()}
    gaps = {
        'rep_rel': np.abs(U['high'] - U['rep_low']),
        'search_rel': np.abs(U['high'] - U['search_low']),
        'rep_exact': np.abs(U['exact'] - U['rep_low']),
        'search_exact': np.abs(U['exact'] - U['search_low']),
        'high_exact': np.abs(U['exact'] - U['high']),
    }
    slopes = {k: ols_slope(v) for k,v in gaps.items()}

    for i,qid in enumerate(MAIN_IDS):
        rec = {
            'query_id': qid,
            'policy': policy['name'],
            'family': policy['family'],
            'alpha': policy['alpha'],
            'rep_rel_H3abs': slopes['rep_rel'][i],
            'search_rel_H3abs': slopes['search_rel'][i],
            'rep_exact_H3abs': slopes['rep_exact'][i],
            'search_exact_H3abs': slopes['search_exact'][i],
            'high_exact_H3abs': slopes['high_exact'][i],
            'rep_rel_terminal_gap': gaps['rep_rel'][i,-1],
            'search_rel_terminal_gap': gaps['search_rel'][i,-1],
            'rep_exact_terminal_gap': gaps['rep_exact'][i,-1],
            'search_exact_terminal_gap': gaps['search_exact'][i,-1],
            'high_exact_terminal_gap': gaps['high_exact'][i,-1],
        }
        for b in ['rep_low','high','search_low','exact']:
            for t in range(H+1):
                rec[f'{b}_ndcg_t{t}'] = U[b][i,t]
        records.append(rec)

    pd.DataFrame(records).to_parquet(OUT/'v032_exact_reference_endpoints_partial.parquet', index=False)
    gc.collect()

endpoints = pd.DataFrame(records)
endpoints.to_parquet(OUT/'v032_exact_reference_endpoints.parquet', index=False)
print('endpoint rows:', len(endpoints), 'elapsed min:', round((time.time()-start_time)/60,2))


In [ ]:
# Cell 12 — Query-level primary analysis and paired bootstrap
numeric_cols = [
    'rep_rel_H3abs','search_rel_H3abs','rep_exact_H3abs','search_exact_H3abs','high_exact_H3abs',
    'rep_rel_terminal_gap','search_rel_terminal_gap','rep_exact_terminal_gap','search_exact_terminal_gap','high_exact_terminal_gap'
]
q = endpoints.groupby('query_id', as_index=False)[numeric_cols].mean()
q['relative_mechanism_contrast'] = q['rep_rel_H3abs'] - q['search_rel_H3abs']
q['exact_mechanism_contrast'] = q['rep_exact_H3abs'] - q['search_exact_H3abs']
q['reference_change_in_mechanism_contrast'] = q['exact_mechanism_contrast'] - q['relative_mechanism_contrast']
q.to_csv(OUT/'v032_query_level_endpoints.csv', index=False)

def paired_bootstrap(values, reps=BOOTSTRAP_REPS, seed=SEED):
    v = np.asarray(values, dtype=np.float64)
    assert np.isfinite(v).all()
    rng = np.random.default_rng(seed)
    n = len(v)
    boots = np.empty(reps, dtype=np.float64)
    chunk = 1000
    pos = 0
    while pos < reps:
        m = min(chunk, reps-pos)
        idx = rng.integers(0, n, size=(m,n))
        boots[pos:pos+m] = v[idx].mean(axis=1)
        pos += m
    return {
        'n_queries': n,
        'mean': float(v.mean()),
        'ci95': [float(np.quantile(boots,0.025)), float(np.quantile(boots,0.975))],
    }

analyses = {}
for col in [
    'rep_exact_H3abs','search_exact_H3abs','exact_mechanism_contrast',
    'rep_rel_H3abs','search_rel_H3abs','relative_mechanism_contrast',
    'reference_change_in_mechanism_contrast','high_exact_H3abs',
    'rep_exact_terminal_gap','search_exact_terminal_gap','high_exact_terminal_gap'
]:
    analyses[col] = paired_bootstrap(q[col].to_numpy())

primary = analyses['exact_mechanism_contrast']
primary['classification'] = (
    'POSITIVE' if primary['ci95'][0] > 0 else
    'NEGATIVE' if primary['ci95'][1] < 0 else
    'UNRESOLVED'
)

result = {
    'study_id':'ARC-v0.32',
    'protocol_sha256': protocol_sha,
    'reference_definition': protocol['reference_definition'],
    'primary_estimand':'representation_minus_search_effort_H3abs_using_exhaustive_reference',
    'primary_direction':'two-sided',
    'primary': primary,
    'secondary': analyses,
    'retuning_performed_after_protocol_freeze': False,
}
(OUT/'v032_primary_gate.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
print(json.dumps(result['primary'], indent=2))


In [ ]:
# Cell 13 — Descriptive consequence: how much does the relative-high comparator leave unresolved?
# This does NOT replace the frozen primary. It quantifies residual approximation of SQ8@64 vs exhaustive search.
round0_residual = {
    'mean_abs_ndcg_gap_sq8_np64_to_exact': float(round0['sq8_np64_abs_ndcg_gap_to_exact'].mean()),
    'median_abs_ndcg_gap_sq8_np64_to_exact': float(round0['sq8_np64_abs_ndcg_gap_to_exact'].median()),
    'mean_top100_overlap_sq8_np64_vs_exact': float(round0['sq8_np64_exact_top100_overlap'].mean()),
    'fraction_exact_ndcg_equal_sq8_np64': float(np.mean(np.isclose(round0['ndcg_exact'], round0['ndcg_sq8_np64'], atol=1e-12))),
}

# Optional diagnostic: one-shot absolute gap vs terminal absolute gap under the exhaustive reference.
# It is descriptive only because policy-events within a query are dependent.
diag = {}
for mech in ['rep','search']:
    x = []
    y = []
    for _, r in endpoints.iterrows():
        low = 'rep_low' if mech == 'rep' else 'search_low'
        one = abs(r['exact_ndcg_t0'] - r[f'{low}_ndcg_t0'])
        terminal = r[f'{mech}_exact_terminal_gap']
        x.append(one); y.append(terminal)
    rho, pval = spearmanr(x, y)
    diag[mech] = {'spearman_round0_abs_gap_vs_terminal_abs_gap': float(rho), 'nominal_p': float(pval)}

consequence = {'round0_residual_high_comparator': round0_residual, 'descriptive_one_shot_vs_terminal': diag}
(OUT/'v032_reference_residual_and_consequence_diagnostics.json').write_text(json.dumps(consequence, indent=2), encoding='utf-8')
print(json.dumps(consequence, indent=2))


In [ ]:
# Cell 14 — Artifact hashes and manuscript-facing conservative wording
artifacts = []
for p in sorted(OUT.iterdir()):
    if p.is_file() and p.name != 'V032_ARTIFACT_SHA256.csv':
        artifacts.append({'file':p.name, 'bytes':p.stat().st_size, 'sha256':sha256_file(p)})
pd.DataFrame(artifacts).to_csv(OUT/'V032_ARTIFACT_SHA256.csv', index=False)

ci = result['primary']['ci95']
cls = result['primary']['classification']
if cls == 'POSITIVE':
    wording = (
        'Under the frozen NQ-GTE post-primary exhaustive-reference audit, the representation-minus-search-effort '
        'H3abs ordering remains positive when both low-fidelity mechanisms are referenced to exhaustive FlatIP search '
        'over the persisted shared corpus-embedding store. This strengthens robustness to the relative-higher-fidelity '
        'comparator choice, but does not create an encoder-original FP32 oracle or a universal ANN theorem.'
    )
elif cls == 'NEGATIVE':
    wording = (
        'The frozen exhaustive-reference audit reverses the representation-minus-search-effort H3abs ordering. '
        'The earlier relative-comparator result should therefore be presented as comparator-conditioned rather than '
        'as an exact-reference mechanism ordering.'
    )
else:
    wording = (
        'The frozen exhaustive-reference audit leaves the representation-minus-search-effort H3abs ordering unresolved. '
        'The earlier relative-comparator mechanism contrast remains valid for its stated operational comparator, but '
        'should not be extrapolated to an exhaustive-reference ordering.'
    )

report = {
    'study_id':'ARC-v0.32',
    'status':'COMPLETE',
    'primary_mean':result['primary']['mean'],
    'primary_ci95':ci,
    'primary_classification':cls,
    'suggested_manuscript_wording':wording,
    'reference_definition':protocol['reference_definition'],
    'protocol_sha256':protocol_sha,
}
(OUT/'v032_final_report.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
print(json.dumps(report, indent=2))
print('\nArtifacts:', OUT)


## Interpretation guardrails

Use the resulting v0.32 evidence only at its stated level.

**If the primary remains positive:** this is reviewer-hardening evidence that the short-horizon mechanism ordering is not solely an artifact of using SQ8/`nprobe=64` as the higher-fidelity branch.

**If unresolved or reversed:** retain that result. The correct paper claim becomes comparator-conditioned; do not retune the exact reference, query subset, policy subset, or horizon.

In all cases, call the new branch **“exhaustive FlatIP over the persisted shared corpus store”** or **“exhaustive-reference retrieval.”** Avoid saying “true encoder oracle” unless a full original FP32 corpus embedding matrix is independently verified and used.
